<a href="https://colab.research.google.com/github/aleenaharoon245-dev/Expense_Tracker/blob/main/a11lab(247)_accessibility_checker.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
#  Accessibility Checker App  (Google Colab ke liye)
#  Is poori file ko Colab ke ek cell mein paste karke Run karo.
#  Zaroorat: sirf ipywidgets (Colab mein pehle se hota hai)
# ============================================================
import re
import datetime
import ipywidgets as w
from IPython.display import display

# ---------------- App ki state (audit report isi se banegi) ----------------
STATE = {
    "theme": "Light",
    "themes_used": set(),
    "invalid_tried": False,
    "form_submitted": False,
    "contrast_checked": False,
    "zoom_used": False,
    "cvd_used": False,
    "fix_viewed": False,
    "log": [],
}

ISSUES = [
    {"id": 1, "type": "Low contrast text", "sev": "High", "status": "Open"},
    {"id": 2, "type": "Image without alt text", "sev": "High", "status": "Open"},
    {"id": 3, "type": "Missing form label", "sev": "Medium", "status": "Open"},
]

# ---------------- Themes ----------------
THEMES = {
    "Light":         {"bg": "#ffffff", "fg": "#1a1a1a", "btn": "#e8eefc", "bd": "#5b6bbf"},
    "Dark":          {"bg": "#1e1e1e", "fg": "#f2f2f2", "btn": "#333333", "bd": "#8ab4f8"},
    "High Contrast": {"bg": "#000000", "fg": "#ffff00", "btn": "#000000", "bd": "#ffffff"},
    "Soft Blue":     {"bg": "#e6f0fa", "fg": "#0b2540", "btn": "#cfe3f7", "bd": "#0b5cab"},
}

style_box = w.HTML()


def apply_theme(name):
    t = THEMES[name]
    STATE["theme"] = name
    STATE["themes_used"].add(name)
    style_box.value = f"""
    <style>
      .app {{ background:{t['bg']}; color:{t['fg']}; padding:14px; border-radius:10px;
              border:3px solid {t['bd']}; font-family:Arial, sans-serif; }}
      .app, .app label, .app .widget-label, .app p, .app td, .app th, .app h3, .app h4, .app b
          {{ color:{t['fg']} !important; }}
      .app button {{ background:{t['btn']} !important; color:{t['fg']} !important;
                     border:2px solid {t['bd']} !important; font-weight:bold; }}
      .app button:focus {{ outline:4px solid #ff9800 !important; }}
      .app table {{ border-collapse:collapse; }}
      .app td, .app th {{ border:1px solid {t['bd']}; padding:6px 10px; text-align:left; }}
    </style>"""


# ---------------- Helper functions ----------------
def hex_to_rgb(h):
    h = h.lstrip("#")
    return tuple(int(h[i:i + 2], 16) for i in (0, 2, 4))


def rgb_to_hex(rgb):
    return "#%02x%02x%02x" % tuple(max(0, min(255, int(round(c)))) for c in rgb)


def luminance(rgb):
    def f(c):
        c /= 255
        return c / 12.92 if c <= 0.03928 else ((c + 0.055) / 1.055) ** 2.4
    r, g, b = [f(c) for c in rgb]
    return 0.2126 * r + 0.7152 * g + 0.0722 * b


def contrast_ratio(c1, c2):
    l1, l2 = luminance(c1), luminance(c2)
    hi, lo = max(l1, l2), min(l1, l2)
    return (hi + 0.05) / (lo + 0.05)


CVD = {
    "Normal vision": None,
    "Protanopia (red-blind)":   [[0.567, 0.433, 0], [0.558, 0.442, 0], [0, 0.242, 0.758]],
    "Deuteranopia (green-blind)": [[0.625, 0.375, 0], [0.7, 0.3, 0], [0, 0.3, 0.7]],
    "Tritanopia (blue-blind)":  [[0.95, 0.05, 0], [0, 0.433, 0.567], [0, 0.475, 0.525]],
    "Grayscale (achromatopsia)": "gray",
}


def simulate(rgb, mode):
    m = CVD[mode]
    if m is None:
        return rgb
    if m == "gray":
        g = 0.299 * rgb[0] + 0.587 * rgb[1] + 0.114 * rgb[2]
        return (g, g, g)
    return tuple(sum(m[i][j] * rgb[j] for j in range(3)) for i in range(3))


def log(msg):
    """Keyboard/mouse action log (login page ke liye)."""
    stamp = datetime.datetime.now().strftime("%H:%M:%S")
    if STATE["log"] and STATE["log"][-1][1] == msg:
        return  # same action baar-baar na likhe
    STATE["log"].append((stamp, msg))
    render_log()


# ============================================================
#  PAGE 1 : THEME + FORM (Login / Sign-up form)
# ============================================================
theme_dd = w.Dropdown(options=list(THEMES), value="Light", description="Theme:")
theme_dd.observe(lambda c: (apply_theme(c["new"]), log(f"Theme changed to {c['new']}")), names="value")

name_in = w.Text(description="Name:", placeholder="e.g. Ali Khan")
email_in = w.Text(description="Email:", placeholder="e.g. ali@gmail.com")
hint = w.HTML("<i>Example email: <b>ali@gmail.com</b></i>")
msg_box = w.HTML()
submit_btn = w.Button(description="Submit Form", icon="check")
log_box = w.HTML()

EMAIL_RE = re.compile(r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$")


def check_email(_=None):
    v = email_in.value.strip()
    if not v:
        hint.value = "<i>Example email: <b>ali@gmail.com</b></i>"
    elif EMAIL_RE.match(v):
        hint.value = "✅ <b>Valid email</b>"
    else:
        hint.value = "❌ <b>Galat email!</b> Format aisa hona chahiye: ali@gmail.com"


name_in.observe(lambda c: log("Typing in Name field (keyboard)"), names="value")
email_in.observe(lambda c: (log("Typing in Email field (keyboard)"), check_email()), names="value")


def on_submit(_):
    log("Clicked 'Submit Form' button")
    problems = []
    if len(name_in.value.strip()) < 2:
        problems.append("Name kam se kam 2 letters ka hona chahiye")
    if not EMAIL_RE.match(email_in.value.strip()):
        problems.append("Email galat hai (example: ali@gmail.com)")
    if problems:
        STATE["invalid_tried"] = True
        msg_box.value = "❌ <b>Form submit nahi hua:</b><ul>" + "".join(f"<li>{p}</li>" for p in problems) + "</ul>"
        log("Submit failed - validation error shown")
    else:
        STATE["form_submitted"] = True
        msg_box.value = f"✅ <b>Form submit ho gaya!</b> Welcome, {name_in.value.strip()}"
        log("Submit success")
        used = sorted({m for _, m in STATE["log"]})
        msg_box.value += "<p><b>Is page par ye actions use hue:</b></p><ul>" + \
                         "".join(f"<li>{u}</li>" for u in used) + "</ul>"


submit_btn.on_click(on_submit)


def render_log():
    rows = "".join(f"<tr><td>{t}</td><td>{m}</td></tr>" for t, m in STATE["log"][-12:])
    log_box.value = "<h4>Action Log (kya-kya click / type kiya)</h4><table><tr><th>Time</th><th>Action</th></tr>" \
                    + (rows or "<tr><td colspan=2>Abhi koi action nahi</td></tr>") + "</table>"


render_log()
page_form = w.VBox([
    w.HTML("<h3>🎨 Theme &amp; Form</h3>"), theme_dd,
    w.HTML("<hr>"), name_in, email_in, hint, submit_btn, msg_box, log_box,
])

# ============================================================
#  PAGE 2 : CONTRAST & ZOOM  (+ Magnification, Color Vision)
# ============================================================
fg_pick = w.ColorPicker(description="Text color", value="#000000")
bg_pick = w.ColorPicker(description="Background", value="#ffffff")
size_sl = w.IntSlider(description="Text size (px)", min=12, max=48, value=16, continuous_update=False)
cvd_dd = w.Dropdown(options=list(CVD), description="Color vision")
sample_txt = w.Text(description="Sample text", value="The quick brown fox jumps over the lazy dog 1234567890")
cz_out = w.HTML()


def render_contrast(_=None):
    fg, bg = hex_to_rgb(fg_pick.value), hex_to_rgb(bg_pick.value)
    r = contrast_ratio(fg, bg)
    STATE["contrast_checked"] = True
    ok = lambda b: "✅ PASS" if b else "❌ FAIL"
    fg2, bg2 = simulate(fg, cvd_dd.value), simulate(bg, cvd_dd.value)
    r2 = contrast_ratio(fg2, bg2)
    zoom = round(size_sl.value / 16 * 100)

    def box(f, b, label, ratio_txt=""):
        return (f"<div style='background:{rgb_to_hex(b)};color:{rgb_to_hex(f)};padding:12px;border:1px solid #888;"
                f"font-size:{size_sl.value}px;margin:6px 0'><small style='font-size:12px'>{label} {ratio_txt}</small><br>"
                f"{sample_txt.value}</div>")

    cz_out.value = f"""
      <h4>Contrast Checker</h4>
      <p><b>Ratio: {r:.2f} : 1</b></p>
      <table>
        <tr><th>Standard</th><th>Needed</th><th>Result</th></tr>
        <tr><td>AA normal text</td><td>4.5 : 1</td><td>{ok(r >= 4.5)}</td></tr>
        <tr><td>AA large text</td><td>3 : 1</td><td>{ok(r >= 3)}</td></tr>
        <tr><td>AAA normal text</td><td>7 : 1</td><td>{ok(r >= 7)}</td></tr>
      </table>
      <h4>Magnification &amp; Color Vision</h4>
      <p>Zoom: <b>{zoom}%</b> &nbsp;|&nbsp; Vision: <b>{cvd_dd.value}</b></p>
      {box(fg, bg, "Normal view", f"(ratio {r:.2f})")}
      {box(fg2, bg2, "As seen with: " + cvd_dd.value, f"(ratio {r2:.2f})")}
    """


def on_zoom(_):
    STATE["zoom_used"] = True
    render_contrast()


def on_cvd(_):
    if cvd_dd.value != "Normal vision":
        STATE["cvd_used"] = True
    render_contrast()


for wd in (fg_pick, bg_pick, sample_txt):
    wd.observe(render_contrast, names="value")
size_sl.observe(on_zoom, names="value")
cvd_dd.observe(on_cvd, names="value")
render_contrast()
STATE["contrast_checked"] = False  # sirf initial render, user ne abhi check nahi kiya

page_contrast = w.VBox([
    w.HTML("<h3>🔍 Contrast &amp; Zoom</h3>"),
    fg_pick, bg_pick, size_sl, cvd_dd, sample_txt, cz_out,
])

# ============================================================
#  PAGE 3 : FIX THE UI  (Before / After + Issue Tracker)
# ============================================================
ba_btn = w.Button(description="Show Before / After", icon="columns")
ba_out = w.HTML()

BEFORE = """
<div style='flex:1;border:2px dashed #c00;padding:10px;background:#fff'>
  <b style='color:#c00'>BEFORE (kharab UI)</b><br><br>
  <span style='color:#bbb;font-size:10px'>Enter your email</span><br>
  <input style='width:90%;border:1px solid #eee;padding:2px' placeholder=''><br><br>
  <button style='font-size:9px;background:#ddd;color:#bbb;border:none'>ok</button>
  <p style='color:#ccc;font-size:10px'>Error shown only in light red</p>
</div>"""
AFTER = """
<div style='flex:1;border:2px solid #080;padding:10px;background:#fff'>
  <b style='color:#060'>AFTER (theek UI)</b><br><br>
  <label for='e' style='color:#111;font-size:16px'>Email address</label><br>
  <input id='e' style='width:90%;border:2px solid #333;padding:8px;font-size:16px' placeholder='ali@gmail.com'><br><br>
  <button style='font-size:16px;padding:8px 18px;background:#0b5cab;color:#fff;border:2px solid #003a75'>Submit</button>
  <p style='color:#900;font-size:15px'>❌ Error: Please enter a valid email</p>
</div>"""
CHANGES = [
    ("Text contrast", "#bbb on white (1.9:1)", "#111 on white (18:1)"),
    ("Font size", "10px", "16px"),
    ("Input label", "Missing", "Visible &lt;label&gt;"),
    ("Button size / color", "Tiny, grey", "Big, high contrast"),
    ("Error message", "Light colour only", "Icon + text + dark colour"),
]


def show_ba(_):
    STATE["fix_viewed"] = True
    rows = "".join(f"<tr><td>{a}</td><td>{b}</td><td>{c}</td></tr>" for a, b, c in CHANGES)
    ba_out.value = f"<div style='display:flex;gap:12px'>{BEFORE}{AFTER}</div>" \
                   f"<h4>Changes</h4><table><tr><th>Item</th><th>Before</th><th>After</th></tr>{rows}</table>"


ba_btn.on_click(show_ba)

# ---- Issue tracker ----
ISSUE_TYPES = ["Low contrast text", "Image without alt text", "Missing form label", "Text too small",
               "No keyboard focus visible", "Error shown by colour only"]
it_type = w.Dropdown(options=ISSUE_TYPES, description="Issue:")
it_sev = w.Dropdown(options=["Low", "Medium", "High"], value="Medium", description="Severity:")
it_add = w.Button(description="Add Issue", icon="plus")
it_pick = w.Dropdown(options=[], description="Select ID:")
it_fix = w.Button(description="Mark as Fixed", icon="check")
it_out = w.HTML()


def render_issues():
    rows = "".join(
        f"<tr><td>{i['id']}</td><td>{i['type']}</td><td>{i['sev']}</td>"
        f"<td>{'✅ Fixed' if i['status'] == 'Fixed' else '🔴 Open'}</td></tr>" for i in ISSUES)
    it_out.value = "<h4>Issue Tracker</h4><table><tr><th>ID</th><th>Issue</th><th>Severity</th><th>Status</th></tr>" \
                   + rows + "</table>"
    it_pick.options = [i["id"] for i in ISSUES if i["status"] == "Open"]


def add_issue(_):
    ISSUES.append({"id": len(ISSUES) + 1, "type": it_type.value, "sev": it_sev.value, "status": "Open"})
    render_issues()


def fix_issue(_):
    for i in ISSUES:
        if i["id"] == it_pick.value:
            i["status"] = "Fixed"
    render_issues()


it_add.on_click(add_issue)
it_fix.on_click(fix_issue)
render_issues()

page_fix = w.VBox([
    w.HTML("<h3>🛠 Fix the UI</h3>"), ba_btn, ba_out,
    w.HTML("<hr>"), it_type, it_sev, it_add, it_pick, it_fix, it_out,
])

# ============================================================
#  PAGE 4 : AUDIT REPORT
# ============================================================
audit_btn = w.Button(description="Refresh Audit Report", icon="refresh")
audit_out = w.HTML()


def build_audit(_=None):
    fg, bg = hex_to_rgb(fg_pick.value), hex_to_rgb(bg_pick.value)
    r = contrast_ratio(fg, bg)
    open_issues = [i for i in ISSUES if i["status"] == "Open"]
    checks = [
        ("Form: galat email par error aata hai",
         "Galat email daalo (abc@) aur Submit dabao",
         STATE["invalid_tried"]),
        ("Form: sahi data se submit hota hai",
         "Naam + sahi email daalo, Submit dabao",
         STATE["form_submitted"]),
        ("Action log (keyboard/click) record hota hai",
         "Form par type karo, log table dekho",
         len(STATE["log"]) > 0),
        ("Theme badalne ka option kaam karta hai",
         "Kam se kam 2 themes try karo (High Contrast zaroor)",
         len(STATE["themes_used"]) >= 2),
        (f"Contrast ratio AA (4.5:1) - abhi {r:.2f}:1",
         "Contrast & Zoom page par colors chuno",
         STATE["contrast_checked"] and r >= 4.5),
        ("Text zoom (magnification) test hua",
         "Text size slider hilao",
         STATE["zoom_used"]),
        ("Color vision simulation test hua",
         "Color vision dropdown mein koi mode chuno",
         STATE["cvd_used"]),
        ("Before / After UI dekha gaya",
         "Fix the UI page par button dabao",
         STATE["fix_viewed"]),
        (f"Saare issues fixed (open: {len(open_issues)})",
         "Issue Tracker mein 'Mark as Fixed' karo",
         len(open_issues) == 0),
    ]
    rows, passed = "", 0
    for name, how, ok in checks:
        passed += bool(ok)
        rows += f"<tr><td>{name}</td><td>{how}</td><td>{'✅ PASS' if ok else '❌ FAIL / Not tested'}</td></tr>"
    pct = round(passed / len(checks) * 100)
    verdict = "ACCESSIBLE ✅" if pct == 100 else ("NEEDS WORK ⚠️" if pct >= 60 else "NOT ACCESSIBLE ❌")
    audit_out.value = f"""
      <h3>📑 Audit Report</h3>
      <p>Date: {datetime.date.today()} &nbsp;|&nbsp; Theme: {STATE['theme']}</p>
      <p><b>Score: {passed}/{len(checks)} ({pct}%) - {verdict}</b></p>
      <table><tr><th>Check</th><th>How to test</th><th>Result</th></tr>{rows}</table>"""


audit_btn.on_click(build_audit)
build_audit()
page_audit = w.VBox([audit_btn, audit_out])

# ============================================================
#  MAIN NAVIGATION (6 buttons)
# ============================================================
content = w.VBox([page_form])
PAGES = [
    ("🎨 Theme & Form", page_form),
    ("🔍 Contrast & Zoom", page_contrast),
    ("🛠 Fix the UI", page_fix),
    ("📑 Audit Report", page_audit),
]


def make_nav(label, page):
    b = w.Button(description=label, layout=w.Layout(width="auto"))

    def go(_):
        if page is page_audit:
            build_audit()
        content.children = [page]
        log(f"Clicked menu: {label}")
    b.on_click(go)
    return b


nav = w.HBox([make_nav(l, p) for l, p in PAGES], layout=w.Layout(flex_flow="row wrap"))
app = w.VBox([w.HTML("<h2>♿ Accessibility Checker</h2>"), nav, w.HTML("<hr>"), content])
app.add_class("app")

apply_theme("Light")
display(style_box, app)

HTML(value='\n    <style>\n      .app { background:#ffffff; color:#1a1a1a; padding:14px; border-radius:10px;\n…